## Basic Multi-LLM Workflows

This notebook demonstrates three simple multi-LLM workflows. They trade off cost or latency for potentially improved task performances:

1. **Prompt-Chaining**: Decomposes a task into sequential subtasks, where each step builds on previous results
2. **Parallelization**: Distributes independent subtasks across multiple LLMs for concurrent processing
3. **Routing**: Dynamically selects specialized LLM paths based on input characteristics

Note: These are sample implementations meant to demonstrate core concepts - not production code.

In [3]:
!pip install anthropic

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [anthropic]/5 [anthropic]


In [2]:
!pip install python-dotenv

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [25]:
import os
import re
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

def llm_call(prompt: str, system_prompt: str = "", model: str = "claude-sonnet-4-5") -> str:
    """
    call model with the given prompt and returns a response
    args:
        prompt: str - the prompt to call the model with
        system_prompt: str - the system prompt to use for the model
        model: str - the model to use for the call
    returns:
        str - the response from the model
    """
    ant_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    messages = [
        {"role": "user", "content": prompt},
    ]
    response = ant_client.messages.create(
        model=model,
        max_tokens=4096,
        system=system_prompt,
        messages=messages,
        temperature=0.1
    )
    return response.content[0].text


def extract_xml(text: str, tag: str) -> str:
    """ extract the content of the specified xml tag from the text"""
    match =  re.search(rf"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return match.group(1) if match else ""


In [15]:
from concurrent.futures import ThreadPoolExecutor
from typing import List, Dict

In [16]:
names = ["John", "Jane", "Jim", "Jill"]
for i, name in enumerate(names, 1):
    print(f"{i}. {name}")


1. John
2. Jane
3. Jim
4. Jill


In [27]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)


In [28]:
logger.debug("debug message")
logger.info("info message")
logger.warning("warning message")
logger.error("error message")
logger.critical("critical message")



2025-12-08 20:36:54,331 - __main__ - INFO - info message
2025-12-08 20:36:54,332 - __main__ - WARNING - warning message
2025-12-08 20:36:54,332 - __main__ - ERROR - error message
2025-12-08 20:36:54,332 - __main__ - CRITICAL - critical message


In [29]:
def chain(input: str, prompts: List[str]) -> str:
    """chain multiple llms calls sequentially, passing results between steps."""
    result = input
    for i, prompt in enumerate(prompts, 1):
        logger.info(f"\nstep {i}:")
        result = llm_call(f"{prompt}\n<input>{result}</input>")
        logger.info(result)
    return result

In [30]:

def parallel(prompt: str, inputs: List[str], n_workers: int = 3) -> List[str]:
    """process multiple inputs concurrently with the same prompt."""
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = [executor.submit(llm_call, f"{prompt}\n<input>{x}</input>") for x in inputs]
        return [f.result() for f in futures]

In [31]:



def route(input: str, routes: Dict[str, str]) -> str:
    """route input to specialized prompt using content classification."""
    # first determine appropriate route using llm with chain-of-thought
    logger.info(f"\navailable routes: {list(routes.keys())}")
    options = "\n".join([f"<option>{k}</option>" for k in routes.keys()])
    selector_prompt = f"""
    <task>
        Analyze the input and select the most appropriate support team from these options: 
            <options>
                {options}
            </options>
        First explain your reasoning, then provide your selection in this XML format:

        <reasoning>
            Brief explanation of why this ticket should be routed to a specific team.
            Consider key terms, user intent, and urgency level.
        </reasoning>

        <selection>
            The chosen team name
        </selection>
    </task>

    <input>{input}</input>""".strip()

    route_response = llm_call(selector_prompt)
    reasoning = extract_xml(route_response, "reasoning")
    route_key = extract_xml(route_response, "selection").strip().lower()

    logger.info("routing analysis:")
    logger.info(reasoning)
    logger.info(f"\nselected route: {route_key}")

    # process input with selected specialized prompt
    selected_prompt = routes[route_key]
    return llm_call(f"{selected_prompt}\n<input>{input}</input>")

## Example Usage

Below are practical examples demonstrating each workflow:
1. Chain workflow for structured data extraction and formatting
2. Parallelization workflow for stakeholder impact analysis
3. Route workflow for customer support ticket handling

In [36]:
# example 1: Chain workflow for structured data extraction and formatting
# each step progressively transforms raw text into a formatted table

data_processing_steps = [
    """
    <task>
        <instruction>Extract only the numerical values and their associated metrics from the text.</instruction>
        <guide>Format each as 'value: metric' on a new line.</guide>
        <example>
            92: customer satisfaction
            45%: revenue growth
        </example>
    </task>
    """,
    """
    <task>
        <instruction>Convert all numerical values to percentages where possible.</instruction>
        <guide>If not a percentage or points, convert to decimal (e.g., 92 points -> 92.00).</guide>
        <guide>If currency is mentioned, convert to decimal (e.g., $43 per user -> 43.00).</guide>
        <guide>Keep the format 'value: metric' on each line.</guide>
        <example>
            92%: customer satisfaction
            45%: revenue growth
        </example>
    </task>
    """,
    """
    <task>
        <instruction>Sort all lines in descending order by numerical value.</instruction>
        <guide>Keep the format 'value: metric' on each line.</guide>
        <guide>The ranking should be the same irrespective of the unit (percentage, points, decimal, etc).</guide>
        <example>
            92%: customer satisfaction
            87%: employee satisfaction
        </example>
    </task>
    """,
    """
    <task>
        <instruction>Format the sorted data as a markdown table with columns.</instruction>
        <example>
            | Metric | Value |
            |:--|--:|   
            | Customer Satisfaction | 92% |
        </example>
    </task>
    """,
]



In [37]:
report = """
Q3 Performance Summary:
Our customer satisfaction score rose to 92 points this quarter.
Revenue grew by 45% compared to last year.
Market share is now at 23% in our primary market.
Customer churn decreased to 5% from 8%.
New user acquisition cost is $43 per user.
Product adoption rate increased to 78%.
Employee satisfaction is at 87 points.
Operating margin improved to 34%.
"""

print("\nInput text:")
print(report)
formatted_result = chain(report, data_processing_steps)

2025-12-08 20:42:11,635 - __main__ - INFO - 
step 1:
2025-12-08 20:42:11,641 - anthropic._base_client - DEBUG - Request options: {'method': 'post', 'url': '/v1/messages', 'timeout': Timeout(connect=5.0, read=600, write=600, pool=600), 'files': None, 'idempotency_key': 'stainless-python-retry-4e703553-491b-48f9-bb24-dbc4af71d848', 'json_data': {'max_tokens': 4096, 'messages': [{'role': 'user', 'content': "\n    <task>\n        <instruction>Extract only the numerical values and their associated metrics from the text.</instruction>\n        <guide>Format each as 'value: metric' on a new line.</guide>\n        <example>\n            92: customer satisfaction\n            45%: revenue growth\n        </example>\n    </task>\n    \n<input>\nQ3 Performance Summary:\nOur customer satisfaction score rose to 92 points this quarter.\nRevenue grew by 45% compared to last year.\nMarket share is now at 23% in our primary market.\nCustomer churn decreased to 5% from 8%.\nNew user acquisition cost is 


Input text:

Q3 Performance Summary:
Our customer satisfaction score rose to 92 points this quarter.
Revenue grew by 45% compared to last year.
Market share is now at 23% in our primary market.
Customer churn decreased to 5% from 8%.
New user acquisition cost is $43 per user.
Product adoption rate increased to 78%.
Employee satisfaction is at 87 points.
Operating margin improved to 34%.



2025-12-08 20:42:14,401 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Mon, 08 Dec 2025 20:42:14 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Content-Encoding', b'gzip'), (b'anthropic-ratelimit-input-tokens-limit', b'30000'), (b'anthropic-ratelimit-input-tokens-remaining', b'30000'), (b'anthropic-ratelimit-input-tokens-reset', b'2025-12-08T20:42:13Z'), (b'anthropic-ratelimit-output-tokens-limit', b'8000'), (b'anthropic-ratelimit-output-tokens-remaining', b'8000'), (b'anthropic-ratelimit-output-tokens-reset', b'2025-12-08T20:42:14Z'), (b'anthropic-ratelimit-requests-limit', b'50'), (b'anthropic-ratelimit-requests-remaining', b'49'), (b'anthropic-ratelimit-requests-reset', b'2025-12-08T20:42:13Z'), (b'retry-after', b'47'), (b'anthropic-ratelimit-tokens-limit', b'38000'), (b'anthropic-ratelimit-tokens-remaining', b'38000'), (b'anthropic-ratelimit-toke

In [40]:
# Example 2: Parallelization workflow for stakeholder impact analysis
# Process impact analysis for multiple stakeholder groups concurrently

stakeholders = [
    """Customers:
    - Price sensitive
    - Want better tech
    - Environmental concerns""",
    """Employees:
    - Job security worries
    - Need new skills
    - Want clear direction""",
    """Investors:
    - Expect growth
    - Want cost control
    - Risk concerns""",
    """Suppliers:
    - Capacity constraints
    - Price pressures
    - Tech transitions""",
]

impact_results = parallel(
    """
    <task>
        <instruction>Analyze how market changes will impact this stakeholder group.</instruction>
        <guide>Provide specific impacts and recommended actions.</guide>
        <guide>Follow the formatting instructions provided in the example.</guide>
        <example>
            <result>
                <stakeholder>Customers</stakeholder>
                <impact>
                    <sections>
                        <section>
                            <title>Impact on Price Sensitivity</title>
                            <priority>High</priority>
                        </section>
                    </sections>
                </impact>
                <actions>
                    <action>
                        <title>Recommended Actions</title>
                        <priority>Medium</priority>
                    </action>
                </actions>
            </result>
        </example>
    </task>
    """,
    stakeholders,
)

for result in impact_results:
    print(result)
    print("+" * 80)

2025-12-08 20:51:49,568 - anthropic._base_client - DEBUG - Request options: {'method': 'post', 'url': '/v1/messages', 'timeout': Timeout(connect=5.0, read=600, write=600, pool=600), 'files': None, 'idempotency_key': 'stainless-python-retry-372d5fe2-64d7-4916-9cd8-3e66982dcffc', 'json_data': {'max_tokens': 4096, 'messages': [{'role': 'user', 'content': '\n    <task>\n        <instruction>Analyze how market changes will impact this stakeholder group.</instruction>\n        <guide>Provide specific impacts and recommended actions.</guide>\n        <guide>Follow the formatting instructions provided in the example.</guide>\n        <example>\n            <result>\n                <stakeholder>Customers</stakeholder>\n                <impact>\n                    <sections>\n                        <section>\n                            <title>Impact on Price Sensitivity</title>\n                            <priority>High</priority>\n                        </section>\n                    </s

```xml
<result>
    <stakeholder>Customers</stakeholder>
    <impact>
        <sections>
            <section>
                <title>Impact on Price Sensitivity</title>
                <priority>High</priority>
                <description>Market changes may lead to increased costs passed to customers through higher prices, potentially reducing affordability and purchase frequency for price-sensitive segments. Economic pressures could intensify budget constraints.</description>
            </section>
            <section>
                <title>Impact on Technology Expectations</title>
                <priority>High</priority>
                <description>Customers will expect continuous innovation and improved features as competitors advance their offerings. Failure to meet evolving tech standards may result in customer attrition to more innovative alternatives.</description>
            </section>
            <section>
                <title>Impact on Environmental Values</title>
  

In [41]:
# Example 3: Route workflow for customer support ticket handling
# Route support tickets to appropriate teams based on content analysis

support_routes = {
    "billing": """
    <task>
        <instruction>You are a billing support specialists:</instruction>
        <instruction>Follow the provided guidelines to respond to the billing issue.</instruction>
        <guidelines>
            <guide>Always start with response tags</guide>
            <guide>First acknowledge the specific billing issue</guide>
            <guide>Explain any charges or discrepancies clearly</guide>
            <guide>List concrete next steps with timeline</guide>
            <guide>End with payment options if relevant</guide>
        </guidelines>
        
        <example>
            <response>
                <introduction>
                    Hello, I'm a billing support specialist.
                </introduction>
                <acknowledgement>
                    I understand your concern about the charge on your credit card.
                </acknowledgement>
                <explanation>
                    The charge of $49.99 was due to the upgrade to the $49.99 plan.
                </explanation>
                <next_steps>
                    <step>
                        I will adjust the charge to $29.99 and send you a new invoice.
                    </step>
                    <step>
                        Please let me know if you have any other questions.
                    </step>
                </next_steps>
                <payment_options>
                    <option>
                        You can pay for the upgrade now or wait until the next billing cycle.
                    </option>
                </payment_options>
            </response>
        </example>
    </task>
    """,
    "technical": """
    <task>
        <instruction>You are a technical support engineer.</instruction>
        <instruction>Follow the provided guidelines to respond to the technical issue.</instruction>
        <guidelines>
            <guide>Always start with response tags</guide>
            <guide>List exact steps to resolve the issue</guide>
            <guide>Include system requirements if relevant</guide>
            <guide>Provide workarounds for common problems</guide>
            <guide>End with escalation path if needed</guide>
        </guidelines>
        
        <example>
            <response>
                <introduction>
                    Hello, I'm a technical support engineer.
                </introduction>
                <acknowledgement>
                    I understand your concern about the issue.
                </acknowledgement>
                <explanation>
                    The issue is due to the system requirements.
                </explanation>
                <steps>
                    <step>
                        Please let me know if you have any other questions.
                    </step>
                </steps>
                <system_requirements>
                    <requirement>
                        The system requires a minimum of 4GB of RAM.
                    </requirement>
                </system_requirements>
                <workarounds>
                    <workaround>
                        If the issue persists, please contact our support team at support@example.com.
                    </workaround>
                </workarounds>
                <escalation_path>
                    If the issue persists, please contact our support team at support@example.com.
                </escalation_path>
            </response>
        </example>
    </task>
    """,
    "account": """
    <task>
        <instruction>You are an account security specialist.</instruction>
        <instruction>Follow the provided guidelines to respond to the account issue.</instruction>
        <guidelines>
            <guide>Always start with response tags</guide>
            <guide>Prioritize account security and verification</guide>
            <guide>Provide clear steps for account recovery/changes</guide>
            <guide>Include security tips and warnings</guide>
            <guide>Set clear expectations for resolution time</guide>
        </guidelines>
        
        <example>
            <response>
                <introduction>
                    Hello, I'm a account security specialist.
                </introduction>
                <verification>
                    <request>
                        Please verify my account information.
                    </request>
                </verification>
                <steps>
                    <step>
                        Please let me know if you have any other questions.
                    </step>
                </steps>
                <security_tips>
                    <tip>
                        Please keep your password secure and change it regularly.
                    </tip>
                </security_tips>
                <expectations>
                    <expectation>
                        Please let me know if you have any other questions.
                    </expectation>
                </expectations>
            </response>
        </example>
    </task>""",
    "product": """
    <task>
        <instruction>You are a product specialist.</instruction>
        <instruction>Follow the provided guidelines to respond to the product issue.</instruction>
        <guidelines>
            <guide>Always start with response tags</guide>
            <guide>Focus on feature education and best practices</guide>
            <guide>Include specific examples of usage</guide>
            <guide>Link to relevant documentation sections</guide>
            <guide>Suggest related features that might help</guide>
        </guidelines>
        <example>
            <response>
                <introduction>
                    Hello, I'm a product specialist.
                </introduction>
                <education>
                    Education about the product.
                </education>
                <best_practices>
                    <practice>
                        Please use the latest version of the product.
                    </practice>
                </best_practices>
                <usage_examples>
                    <usage_example>
                        Please use the latest version of the product.
                    </usage_example>
                </usage_examples>
                <documentation_links>
                    <link>
                        Please use the latest version of the product.
                    </link>
                </documentation_links>
                <related_features>
                    <feature>
                        Please use the latest version of the product.
                    </feature>
                </related_features>
            </response>
        </example>
    </task>
    """,
}

# Test with different support tickets
tickets = [
    """<subject>Can't access my account</subject>
    <message>Hi, I've been trying to log in for the past hour but keep getting an 'invalid password' error. 
    I'm sure I'm using the right password. Can you help me regain access? This is urgent as I need to 
    submit a report by end of day
    - John</message>""",
    """<subject>Unexpected charge on my card</subject>
    <message>Hello, I just noticed a charge of $49.99 on my credit card from your company, but I thought
    I was on the $29.99 plan. Can you explain this charge and adjust it if it's a mistake?
    Thanks,
    Sarah</message>""",
    """<subject>How to export data?</subject>
    <message>I need to export all my project data to Excel. I've looked through the docs but can't
    figure out how to do a bulk export. Is this possible? If so, could you walk me through the steps?
    Best regards,
    Mike</message>""",
]

logger.info("processing support tickets...\n")
for i, ticket in enumerate(tickets, 1):
    logger.info(f"\nticket {i}:")
    logger.info("-" * 40)
    logger.info(ticket)
    logger.info("\nresponse:")
    logger.info("-" * 40)
    response = route(ticket, support_routes)
    logger.info(response)
    logger.info("*" * 80)

2025-12-08 21:03:54,993 - __main__ - INFO - processing support tickets...

2025-12-08 21:03:54,994 - __main__ - INFO - 
ticket 1:
2025-12-08 21:03:54,994 - __main__ - INFO - ----------------------------------------
2025-12-08 21:03:54,994 - __main__ - INFO - <subject>Can't access my account</subject>
    <message>Hi, I've been trying to log in for the past hour but keep getting an 'invalid password' error. 
    I'm sure I'm using the right password. Can you help me regain access? This is urgent as I need to 
    submit a report by end of day
    - John</message>
2025-12-08 21:03:54,995 - __main__ - INFO - 
response:
2025-12-08 21:03:54,995 - __main__ - INFO - ----------------------------------------
2025-12-08 21:03:54,995 - __main__ - INFO - 
available routes: ['billing', 'technical', 'account', 'product']
2025-12-08 21:03:55,002 - anthropic._base_client - DEBUG - Request options: {'method': 'post', 'url': '/v1/messages', 'timeout': Timeout(connect=5.0, read=600, write=600, pool=600),